In [1]:
import sys
from pathlib import Path

# Add pub_worm directory to the Python path
sys.path.insert(0, str(Path.cwd().parent))

# Test where we are loading the pub_worm objects from
import inspect
import wormcat3

module = inspect.getmodule(wormcat3)
if hasattr(module, "__file__"):
    print("wormcat3 imported from:", module.__file__)
else:
    print("Could not determine the file path.")

wormcat3 imported from: /Users/dan/Code/Python/wormcat3/wormcat3/__init__.py


# WormCat3: Python Implementation of WormCat.com

**WormCat3** is a Python-based backend implementation of the **WormCat** platform, a tool designed for functional enrichment analysis of *Caenorhabditis elegans* gene sets.

- This implementation powers the [WormCat website](http://wormcat.com), providing a fast, flexible, and extensible platform for analyzing gene function using customizable annotation categories.

> **Note:** You do **not** need to download or install any software to use WormCat3 — it is freely available at [wormcat.com](http://wormcat.com).

## Why Run Your Own Instance of WormCat?

You might want to run your own local or integrated version of WormCat3 if:

1. You want to incorporate WormCat into a larger bioinformatics pipeline.
2. You have a large number of gene sets and prefer not to run them manually via the website.
3. You want to supply your own annotation file or modify an existing one.
4. You want to use extended functionality such as Gene Set Enrichment Analysis (GSEA).
5. You need to convert WormCat annotations into GMT format for use with other tools.
-----

## Setting Up WormCat3: Create a Conda Environment

To ensure reproducibility and avoid dependency conflicts, it's best to run WormCat3 in a dedicated **Conda environment**. 

- Conda environments isolate your project's packages and Python version from the rest of your system.

- We recommend installing **Miniforge3**, a minimal, community-maintained Conda installer that supports cross-platform environments and uses the conda-forge ecosystem by default.

- If you have not already done so. ➡️ Install Miniforge3 from the official source: [conda-forge.org](https://conda-forge.org/download/)

<br>

### Create and activate the WormCat3 environment:

```bash
# Create a new environment with Python 3.12
conda create -n wormcat3 python=3.12

# Activate the environment
conda activate wormcat3

# This is a dependency for GSEA
conda install -c conda-forge rust

pip install wormcat3
```
<br>


- ✅ That’s it — your environment is ready!
- Select the newly installed environment / Kernel in this Notebook (_In most environments it is available for selection in the upper right conner_)
- Now let’s dive in and see how easy it is to explore C. elegans functional enrichment with **WormCat3**.

### Check what version of WormCat3 is installed

In [2]:
import os
import wormcat3
print(f"{wormcat3.__version__=}")
print(os.getcwd())
# Note: This notebook was created using version 0.1.2. Any version 0.1.2 or later should be compatible.

wormcat3.__version__='0.1.9'
/Users/dan/Code/Python/wormcat3/notebooks


### Helper Function: Download Example Data ⬇️

- This helper function simplifies downloading example data from the **WormCat3 GitHub** repository.
- It is not required to run WormCat3 but can be used to quickly obtain example datasets for testing or demonstration purposes.

In [3]:
import os
import urllib.request

def download_example_data(from_path: str, dest_dir_path: str):
    """
    Downloads a file from the wormcat3/example_data GitHub directory to a local destination.
    """
    base_url = "https://raw.githubusercontent.com/DanHUMassMed/wormcat3/main/example_data"
    url = f"{base_url}/{from_path}"

    file_name = os.path.basename(from_path)
    dir_name = os.path.dirname(from_path)
    
    # Ensure the destination directory exists
    dest_dir_path = os.path.join(dest_dir_path, dir_name)
    os.makedirs(dest_dir_path, exist_ok=True)
    
    # Add the file_name to the destination
    dest_file_path = os.path.join(dest_dir_path, file_name)

    # Download the file
    urllib.request.urlretrieve(url, dest_file_path)

    print(f"Downloaded {file_name} to {dest_file_path}")

----
### Example 1: ⬇️ Download an Example Gene Set File

- To run a WormCat3 enrichment analysis, your input must be either a **CSV file** or a **Python list**.
- Unlike the online version, which requires stricter formatting, the **local version accepts CSV files with any number of columns**.
- **Only the first column** is used for analysis; all other columns are ignored.
- **Important:** The column header is *not* used. WormCat3 examines the first few data rows to determine whether the input contains **WormBase Gene IDs** or **Sequence IDs**.
- This example file includes **WormBase Gene IDs** in the first column.

<br>

> **Note:** This file is the output of a DESeq2 analysis and can be used directly as input to WormCat, simplifying integration into bioinformatics pipelines.

In [4]:
import pandas as pd

# Let's download an example file to work with
from_file_path  = "DESeq2_Output/DESeq2-EC-ES/EC-ES-Down.csv"
example_path = f"{os.getcwd()}/example_data/"
download_example_data(from_file_path, example_path)
example1_file_path=f"{example_path}/{from_file_path}"

# Let's take a quick look at the content
example1_df = pd.read_csv(example1_file_path)
example1_df.head()

# This file is the output of a DeSeq2 analysis

Downloaded EC-ES-Down.csv to /Users/dan/Code/Python/wormcat3/notebooks/example_data/DESeq2_Output/DESeq2-EC-ES/EC-ES-Down.csv


,ID,EC_2,EC_3,EC_4,EC_5,ES_2,ES_3,ES_4,ES_5,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,foldChange,log10padj
0,WBGene00000005,185.527916,91.684976,100.492727,130.377809,66.115069,34.593194,46.983949,37.985825,86.720183,-1.449663,0.352212,16.500954,4.862560e-05,6.024024e-04,0.366107,3.220113
1,WBGene00000008,7.135689,19.883489,2.955668,16.514522,0.000000,0.000000,0.000000,0.000000,5.811171,-6.044657,1.597262,13.100003,2.952953e-04,2.783435e-03,0.015149,2.555419
2,WBGene00000112,3262.199198,3274.147818,2923.156082,2717.073537,1333.635386,1490.652183,1337.163188,1328.599443,2208.328354,-1.149006,0.092389,151.423479,8.468968e-35,3.727522e-32,0.450936,31.428580
3,WBGene00000115,5167.428187,4612.969395,5979.317243,4030.412666,1795.496366,1378.486372,1379.448742,1743.730242,3260.911152,-1.651754,0.149496,116.255314,4.178827e-27,1.298304e-24,0.318253,23.886624
4,WBGene00000116,530.419556,416.448626,656.158392,272.924213,182.288689,101.683025,146.589921,78.684923,298.149668,-1.880583,0.337417,29.313700,6.155894e-08,1.639330e-06,0.271574,5.785334


### Example 1: 🧪 Analyze and Visualize Enrichment (Default Options)

- This example runs an enrichment analysis using WormCat3’s default parameters.
- By default, results are saved in a `wormcat_out` directory within your current working directory.
- Inside `wormcat_out`, a subdirectory is created using the provided **title** along with a unique 5-digit random suffix.
- The random suffix helps avoid filename conflicts, especially when opening multiple results in tools like Excel.

<br>

> **Note:** The `run_params_???.txt` file (located in the output folder) records the settings used in the analysis (including the defaults):
> - **Annotation File:** *whole_genome_v2_nov-11-2021.csv*
> - **Significance Method:** *bonferroni*
> - **Significance Threshold:** *0.05*

In [5]:
# Wormcat3 can easily be run from a Jupyter Notebook
from wormcat3 import Wormcat

wormcat = Wormcat(title="My First Run")
wormcat.analyze_and_visualize_enrichment(example1_file_path)
    

genes_in_both=157, gene_set_size=590, category_size=6343, background_size=31389
Contingency Table: a=157, b=433, c=6186, d=24613
genes_in_both=89, gene_set_size=590, category_size=1601, background_size=31389
Contingency Table: a=89, b=501, c=1512, d=29287
genes_in_both=66, gene_set_size=590, category_size=901, background_size=31389
Contingency Table: a=66, b=524, c=835, d=29964
genes_in_both=52, gene_set_size=590, category_size=833, background_size=31389
Contingency Table: a=52, b=538, c=781, d=30018
genes_in_both=51, gene_set_size=590, category_size=495, background_size=31389
Contingency Table: a=51, b=539, c=444, d=30355
genes_in_both=40, gene_set_size=590, category_size=3200, background_size=31389
Contingency Table: a=40, b=550, c=3160, d=27639
genes_in_both=31, gene_set_size=590, category_size=394, background_size=31389
Contingency Table: a=31, b=559, c=363, d=30436
genes_in_both=25, gene_set_size=590, category_size=1188, background_size=31389
Contingency Table: a=25, b=565, c=1163

### Example 1: 📁 WormCat3 Output Files

| Output File                                       | Description |
|--------------------------------------------|-------------|
| category_1_fisher_My_First_Run_65580.csv   | Fisher's exact test results for Category 1 |
| category_1_padj_My_First_Run_65580.csv     | Adjusted p-values (padj) for Category 1    |
| category_1_padj_My_First_Run_65580.svg     | SVG bar chart visualization for Category 1 |
| category_2_fisher_My_First_Run_65580.csv   | Fisher's exact test results for Category 2 |
| category_2_padj_My_First_Run_65580.csv     | Adjusted p-values (padj) for Category 2    |
| category_2_padj_My_First_Run_65580.svg     | SVG bar chart visualization for Category 2 |
| category_3_fisher_My_First_Run_65580.csv   | Fisher's exact test results for Category 3 |
| category_3_padj_My_First_Run_65580.csv     | Adjusted p-values (padj) for Category 3    |
| category_3_padj_My_First_Run_65580.svg     | SVG bar chart visualization for Category 3 |
| genes_not_annotated_My_First_Run_65580.csv | Genes in the input set NOT used in the enrichment  |
| input_annotated_My_First_Run_65580.csv     | Input gene set with assigned annotations   |
| run_params_My_First_Run_65580.txt          | The Date and Parameters used during the run             |
| sunburst_My_First_Run_65580.html           | Interactive sunburst visualization   |


Note: **run_params_My_First_Run_65580.txt** will show what default option where taken

------

### Example 2: 🧬 A note on Annotation Files

- To find a list of available Annotation Files use: `AnnotationsManager.available_annotation_files()`


#### Custom Annotation Files

To use your own annotation files with WormCat3:

- Set the environment variable: `WORMCAT_DATA_PATH` to the path of a directory containing your custom annotation files:
  
  ```bash
  export WORMCAT_DATA_PATH=/path/to/your/annotations
  ```

- Files in this directory must be in CSV format (e.g., whole_genome_v2_nov-11-2021.csv).
- Each file must contain the following header:
    - `Sequence ID,Wormbase ID,Category 1,Category 2,Category 3,Automated Description`

<br>

> **Note:** If WORMCAT_DATA_PATH is set `AnnotationsManager.available_annotation_files()` will return any additional annotations found here.



In [6]:
# To find a list of Available Annotation files
from wormcat3 import AnnotationsManager
AnnotationsManager.available_annotation_files()

['ORF_only_v2_nov-11-2021.csv',
 'ahringer_v2_nov-11-2021.csv',
 'orfeome_v2_nov-11-2021.csv',
 'whole_genome_v2_nov-11-2021.csv']

### Example 2: 🧪 Analyze and Visualize Enrichment with additional options 

This example demonstrates how to run an enrichment analysis using specific parameters:

- **Significance Method:** `fdr_bh` (Benjamini-Hochberg FDR)  
    -  *(Valid options: `fdr_bh`, `bonferroni`)*
- **Significance Threshold:** `0.10`  
    - *(Must be ≥ 0 and < 1)*
- **Annotation File:** `whole_genome_v2_nov-11-2021.csv`

<br>

> **Note:** We also demonstrate `WormcatError` *Exception Management* that should be used when integrating into a bioinfomatics pipeline

In [7]:
# Wormcat can now easily be run from a Jupyter Notebook
from wormcat3 import Wormcat, WormcatError, PAdjustMethod

wormcat = Wormcat(title="My Second Run", annotation_file_name='whole_genome_v2_nov-11-2021.csv')

try:
    wormcat.analyze_and_visualize_enrichment(example1_file_path, p_adjust_method=PAdjustMethod.FDR, p_adjust_threshold=0.10)
except WormcatError as err:
    print(err)

genes_in_both=157, gene_set_size=590, category_size=6343, background_size=31389
Contingency Table: a=157, b=433, c=6186, d=24613
genes_in_both=89, gene_set_size=590, category_size=1601, background_size=31389
Contingency Table: a=89, b=501, c=1512, d=29287
genes_in_both=66, gene_set_size=590, category_size=901, background_size=31389
Contingency Table: a=66, b=524, c=835, d=29964
genes_in_both=52, gene_set_size=590, category_size=833, background_size=31389
Contingency Table: a=52, b=538, c=781, d=30018
genes_in_both=51, gene_set_size=590, category_size=495, background_size=31389
Contingency Table: a=51, b=539, c=444, d=30355
genes_in_both=40, gene_set_size=590, category_size=3200, background_size=31389
Contingency Table: a=40, b=550, c=3160, d=27639
genes_in_both=31, gene_set_size=590, category_size=394, background_size=31389
Contingency Table: a=31, b=559, c=363, d=30436
genes_in_both=25, gene_set_size=590, category_size=1188, background_size=31389
Contingency Table: a=25, b=565, c=1163

### Example 2: 📁 Output

- Allowing customization of the **Significance Method** and **Significance Threshold** provides greater flexibility and precision in experimental design
    - Empowering users to tailor statistical criteria to the specific needs of their study.

-----

### Example 3: ⬇️ Download Background Gene Set File

- To run a WormCat3 enrichment analysis, your background must be a **CSV file** or a Python list of gene IDs.


In [8]:
import os
# Let's download an example file to work with
from_file_path  = "DESeq2_Output/DESeq2-EC-ES/EC-ES_All_detected.csv"
example_path = f"{os.getcwd()}/example_data/"
download_example_data(from_file_path, example_path)
all_detected_file_path = f"{example_path}/{from_file_path}"

Downloaded EC-ES_All_detected.csv to /Users/dan/Code/Python/wormcat3/notebooks/example_data/DESeq2_Output/DESeq2-EC-ES/EC-ES_All_detected.csv



### Example 3: 🧪 Analyze and Visualize Enrichment with Background Gene Set

- This example demonstrates how to run an enrichment analysis using a background Gene Set


In [9]:
from wormcat3 import Wormcat, WormcatError, PAdjustMethod

wormcat = Wormcat(title="My Third Run", annotation_file_name='whole_genome_v2_nov-11-2021.csv')

try:
    results_df = wormcat.analyze_and_visualize_enrichment(example1_file_path, all_detected_file_path, p_adjust_method=PAdjustMethod.FDR, p_adjust_threshold=0.10) # type: ignore
except WormcatError as err:
    print(err)

genes_in_both=157, gene_set_size=590, category_size=4618, background_size=16395
Contingency Table: a=157, b=433, c=4461, d=11344
genes_in_both=89, gene_set_size=590, category_size=1513, background_size=16395
Contingency Table: a=89, b=501, c=1424, d=14381
genes_in_both=66, gene_set_size=590, category_size=825, background_size=16395
Contingency Table: a=66, b=524, c=759, d=15046
genes_in_both=52, gene_set_size=590, category_size=667, background_size=16395
Contingency Table: a=52, b=538, c=615, d=15190
genes_in_both=51, gene_set_size=590, category_size=408, background_size=16395
Contingency Table: a=51, b=539, c=357, d=15448
genes_in_both=40, gene_set_size=590, category_size=1545, background_size=16395
Contingency Table: a=40, b=550, c=1505, d=14300
genes_in_both=31, gene_set_size=590, category_size=344, background_size=16395
Contingency Table: a=31, b=559, c=313, d=15492
genes_in_both=25, gene_set_size=590, category_size=1120, background_size=16395
Contingency Table: a=25, b=565, c=1095

### Example 3: 📁 WormCat3 Additional Output Files

- When running with a background gene set two additional outputs are created

| Output File                                       | Description |
|--------------------------------------------|-------------|
| background_annotated_My_Third_Run_96105.csv   | Background gene set with assigned annotations |
| background_not_annotated_My_Third_Run_96105.csv   | Background Genes NOT used in the enrichment Analysis  |


---

### Example 4: ⬇️ Download and Analyze Multiple Gene Sets (WormCat3 Batch)

You can run a WormCat3 Batch using one of the following input formats:

- An **Excel file** where each sheet contains a separate gene set.
- A **directory of CSV files**, where each CSV represents an individual gene set to be processed.

<br>

- Excel file requirements:
    - The Spreadsheet Name should ONLY be composed of Letters, Numbers, and Underscores (_) and has an extension .xlsx, .xlt, .xls
    - The individual Sheet Names (i.e., Tab name) within the spreadsheet should ONLY be composed of Letters, Numbers, and Underscores (_).
    - Each Sheet requires that the first column be 'Sequence ID' or 'Wormbase ID'
- CSV file requirements:
    - The CSV File Name should ONLY be composed of Letters, Numbers, and Underscores (_) and has an extension .csv
    - Each CSV file requires that the first column be 'Sequence ID' or 'Wormbase ID'
    - Note: CSV files can have more that one column however only the first column is used


In [ ]:
# Let's download an example file to work with
from_file_path  = "Murphy_TS.xlsx"
example_path = f"{os.getcwd()}/example_data/"
download_example_data(from_file_path, example_path)
batch_excel_path=f"{example_path}/{from_file_path}"


### Example 4: 🧪 **Batch** Analyze and Visualize Enrichment

- This example demonstrates how to run Batch an enrichment analysis using a multiple Gene Sets
- The Gene Sets are placed in an Excel file
    - The Sheet name is used as the title for each individual run

In [ ]:
# Wormcat can now easily be run from a Jupyter Notebook
from wormcat3 import Wormcat, PAdjustMethod

wormcat = Wormcat(title="Murphy_TS")
wormcat.wormcat_batch(batch_excel_path, p_adjust_method=PAdjustMethod.FDR)

### Example 4: 📁 Output

- The output of a batch run creates a directory for each individual gene set
- Additionally there is a summary Excel file created that aggregates all the results

| Output File              | Description                                      |
|--------------------------|--------------------------------------------------|
| Murphy_TS_CSVs           | Folder containing Excel sheets as CSV outputs    |
| intestine_44951          | CSV file for intestine-specific gene data        |
| hypodermis_58109         | CSV file for hypodermis-specific gene data       |
| neurons_85298            | CSV file for neuron-specific gene data           |
| muscle_93148             | CSV file for muscle-specific gene data           |
| Murphy_TS_48591.xlsx     | Combined Excel file summarizing all results      |

---

### Example 5: ⬇️ Analyze Multiple Gene Sets from CSV Files (WormCat3 Batch)

- In this example, we extract individual sheets from the Excel file above and save them as separate CSV files.
- **Note:** This step is optional. What *is* required is a directory containing CSV files, each with gene identifiers in the **first column**.

In [ ]:
from wormcat3.wormcat_excel import WormcatExcel 
csv_file_path = f"{os.getcwd()}/example_data/csv_files"
WormcatExcel.extract_csv_files(batch_excel_path, csv_file_path)

### Example 5: 🧪 **Batch** Analyze and Visualize Enrichment

- This example demonstrates how to perform batch enrichment analysis using multiple gene sets.
- Each gene set is saved as a separate CSV file within a directory.
  - The name of each file is used as the title for its corresponding analysis run.

In [ ]:
# Wormcat can now easily be run from a Jupyter Notebook
from wormcat3 import Wormcat

wormcat = Wormcat(title="Murphy_TS_CSV")
wormcat.wormcat_batch(csv_file_path)

### Example 5: 📁 Output

- The output of a batch run creates a directory for each individual gene set
- Additionally there is a summary Excel file created that aggregates all the results

| Output File              | Description                                      |
|--------------------------|--------------------------------------------------|
| intestine_44951          | CSV file for intestine-specific gene data        |
| hypodermis_58109         | CSV file for hypodermis-specific gene data       |
| neurons_85298            | CSV file for neuron-specific gene data           |
| muscle_93148             | CSV file for muscle-specific gene data           |
| Murphy_TS_48591.xlsx     | Combined Excel file summarizing all results      |

<br>

> **Note:** When Batch WormCat3 is run with a CSV directory you do not get the CSV files in the output

---

### Example 6: ⬇️ Download Example Gene Set File for GSEA

- In this example, we use the alldetected results from a DESeq2 experiment


In [ ]:
# Let's download an example file to work with
from_file_path  = "DESeq2_Output/DESeq2-EC-ES/EC-ES_All_detected.csv"
example_path = f"{os.getcwd()}/example_data/"
download_example_data(from_file_path, example_path)
all_detected_file_path=f"{example_path}/{from_file_path}"

### Example 6: 🧪 Gene Set File for GSEA

- Input CSV must have three columns `'ID', 'log2FoldChange', 'pvalue'`
- Other columns are ignored
- The ranking metric is: $\text{Rank} = \operatorname{sign}(\log_2 \text{FoldChange}) \times -\log_{10}(p\text{-value})$

In [ ]:
from wormcat3 import Wormcat

wormcat = Wormcat(title="EC-ES-GSEA")
wormcat.perform_gsea_analysis(all_detected_file_path)

### Example 6: 📁 Output

- The output of a GSEA run creates a directory for each Wormcat Category


| Output File                                | Description                                                   |
|--------------------------------------------|---------------------------------------------------------------|
| gsea_category_1_EC-ES-GSEA_02114           | GSEA result summary for category 1                            |
| gsea_category_2_EC-ES-GSEA_02114           | GSEA result summary for category 2                            |
| gsea_category_3_EC-ES-GSEA_02114           | GSEA result summary for category 3                            |
| gsea_category_1_EC-ES-GSEA_02114.csv       | Raw GSEA data for category 1 (CSV format)                     |
| gsea_category_2_EC-ES-GSEA_02114.csv       | Raw GSEA data for category 2 (CSV format)                     |
| gsea_category_3_EC-ES-GSEA_02114.csv       | Raw GSEA data for category 3 (CSV format)                     |
| genes_removed_from_analysis_EC-ES-GSEA_02114.csv | Genes excluded from analysis due to missing annotations |
| run_params_EC-ES-GSEA_02114.txt            | Text file containing parameters used during the analysis run  |

<br>
<br> 

> **Note:** 
> <small>
> - **Term:** The name of the gene set or functional category being evaluated for enrichment (e.g., "Transcription: chromatin structure: histone").
> - **FDR (False Discovery Rate):** The estimated probability that this enrichment is a false positive after correcting for multiple hypothesis testing.
> - **ES (Enrichment Score):** A measure of how much the genes in the set are overrepresented at the top or bottom of the ranked gene list.
> - **NES (Normalized Enrichment Score):** The ES normalized to account for gene set size, allowing comparisons across different gene sets.
> - **P-value:** The probability that the observed ES could have occurred by chance based on random permutations of the data.
> - **Tag %:** The number of genes from the gene set contributing to the enrichment score over the total size of the gene set (e.g., 5 out of 65 genes).
> </small>


-----

### Utility Features

### Extract Annotations to GMT format

- WormCat3 Annotations can be extracted into The Gene Matrix Transposed (GMT) file format.
    - The GMT format was developed by the Broad Institute as part of their Gene Set Enrichment Analysis (GSEA) software suite.
    - The GMT format is a widely adopted standard in bioinformatics for representing gene sets. 
    - Extracting to GMT allows you to bring Wormcat annotations to other tools
 ￼

In [ ]:
from wormcat3 import AnnotationsManager

annotations_manager = AnnotationsManager()
output_dir_path="./wormcat_out"
annotations_manager.create_gmt_for_annotations(output_dir_path)


## <a name="citation"></a>Citation

##### If you use WormCat in a published work please cite:

> **WormCat**: an online tool for annotation and visualization of Caenorhabditis elegans genome-scale data
>
> Amy D Holdorf, Daniel P Higgins, Anne C. Hart, Peter R Boag, Gregory Pazour, Albertha J. M. Walhout, Amy Karol Walker
>
> [GENETICS February 1, 2020 vol. 214 no. 2 279-294;](https://academic.oup.com/genetics/article/214/2/279/5930455?login=false)

--------
